# module-modules-iter-isinstance-dispatch — ex1: count + tag layers by type using model.modules() + isinstance

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-modules-iter-isinstance-dispatch`. Running the final beacon cell reports progress against the `GAN: model.modules() isinstance dispatch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.modules() isinstance dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-modules-iter-isinstance-dispatch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-modules-iter-isinstance-dispatch"
DD_SUBTOPIC = "GAN: model.modules() isinstance dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## model.modules() + isinstance dispatch — quick refresher

`model.modules()` is an iterator over EVERY submodule in the network (including the model itself, recursively). Pair it with `isinstance` to do per-layer-type work without `apply`:

```python
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)
```

**`modules()` vs `apply(fn)`.** `apply` calls `fn(m)` on every submodule too — but `modules()` gives you a plain Python loop where you can branch, count, accumulate stats, or break early. Use `apply` for pure transforms; use `modules()` when you want a procedural body.

**`modules()` vs `children()`.** `children()` is shallow — only the DIRECT submodules. `modules()` is deep — every descendant. For init you almost always want `modules()`.

### Exercise 1 — count + tag layers by type using model.modules() + isinstance

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `model.modules()` plus `isinstance` dispatch to walk every submodule of a network and accumulate a count per layer type into a dictionary.
> Keywords: modules, isinstance, dispatch, inspection
> ```

**KCs targeted:** `modules-iter-recursive`, `isinstance-layer-dispatch`

Implement `ex1_count_layer_types(model)`. The procedural-loop version of `model.apply` — useful when you need branch + branch + accumulator, not a pure transform:

1. Initialize a dict `counts` with keys `'conv2d'`, `'convtranspose2d'`, `'batchnorm'`, `'linear'`, `'other'`, all set to 0.
2. Iterate `for m in model.modules()`. (`modules()` is recursive — it yields the model itself plus every nested submodule.)
3. Dispatch by `isinstance`:
   - `nn.Conv2d` → bump `'conv2d'`
   - `nn.ConvTranspose2d` → bump `'convtranspose2d'`
   - `nn.BatchNorm1d` or `nn.BatchNorm2d` → bump `'batchnorm'`
   - `nn.Linear` → bump `'linear'`
   - any other type → bump `'other'`
4. Return `counts`.

Important: each module gets counted EXACTLY ONCE. The branches must be mutually exclusive — use `if / elif`, not stacked `if`s.

Input: `model` — `nn.Module`.
Output: dict[str, int] with the five keys above.

The visualization runs your counter on a DCGAN-shaped model and renders the layer-type counts as a horizontal bar chart.

In [ ]:
def ex1_count_layer_types(model: nn.Module) -> dict:
    counts = {'conv2d': 0, 'convtranspose2d': 0, 'batchnorm': 0, 'linear': 0, 'other': 0}
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            counts['conv2d'] += 1
        elif isinstance(m, nn.ConvTranspose2d):
            counts['convtranspose2d'] += 1
        elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            counts['batchnorm'] += 1
        elif isinstance(m, nn.Linear):
            counts['linear'] += 1
        else:
            counts['other'] += 1
    return counts


<details><summary>Solution</summary>

```python
def ex1_count_layer_types(model: nn.Module) -> dict:
    counts = {'conv2d': 0, 'convtranspose2d': 0, 'batchnorm': 0, 'linear': 0, 'other': 0}
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            counts['conv2d'] += 1
        elif isinstance(m, nn.ConvTranspose2d):
            counts['convtranspose2d'] += 1
        elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            counts['batchnorm'] += 1
        elif isinstance(m, nn.Linear):
            counts['linear'] += 1
        else:
            counts['other'] += 1
    return counts
```

**`model.modules()` is recursive.** It yields the model itself FIRST, then every descendant — depth-first. For `nn.Sequential(A, B)` you get `Sequential, A, B`, in that order. That's why the model + every container shows up under `'other'`.

**`Conv2d` is NOT a subclass of `ConvTranspose2d`.** They're siblings under `_ConvNd`. So you can use `if / elif` without worrying about one matching the other. (If you ever subclass Conv2d, mind the ordering — check the more specific class first.)

**Why `elif`, not stacked `if`s.** Mutually exclusive dispatch — every module gets counted exactly once. Stacked `if`s would double-count a hypothetical subclass that satisfies two branches.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()